# 02 — Developer-Profile Salary Model

Train XGBoost on SO 2025 survey data. Compute actual salary ranges (p25/median/p75) per profile bucket.

In [1]:
import os, warnings
warnings.filterwarnings("ignore")
os.chdir("/app")
import pathlib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import LabelEncoder
import joblib
from xgboost import XGBRegressor

DATA_PROC = pathlib.Path("data/processed")
MODELS = pathlib.Path("models")
MODELS.mkdir(exist_ok=True)


## Feature Engineering

In [2]:
so25 = pd.read_parquet(DATA_PROC / "so25_processed.parquet")
print(f"Loaded SO 2025: {so25.shape}")

# Primary role (first DevType)
so25["DevType_primary"] = so25["DevType"].str.split(";").str[0].str.strip()

# Top 15 roles (enough samples per role)
top_roles = (
    so25.groupby("DevType_primary")["CompUSD"]
    .count()
    .nlargest(15)
    .index.tolist()
)
so25 = so25[so25["DevType_primary"].isin(top_roles)].copy()
print(f"After role filter: {len(so25):,} rows, {len(top_roles)} roles")
print("Roles:", top_roles)


Loaded SO 2025: (13739, 175)
After role filter: 12,423 rows, 15 roles
Roles: ['Developer, full-stack', 'Developer, back-end', 'Architect, software or solutions', 'Developer, desktop or enterprise applications', 'Developer, front-end', 'Engineering manager', 'Developer, embedded applications or devices', 'DevOps engineer or professional', 'Other (please specify):', 'Developer, mobile', 'Data engineer', 'Academic researcher', 'Data scientist', 'AI/ML engineer', 'Senior executive (C-suite, VP, etc.)']


In [3]:
# Education level → ordinal
ED_ORDER = {
    "Primary/elementary school": 0,
    "Secondary school (e.g. American high school, German Realschule or Gymnasium, etc.)": 1,
    "Some college/university study without earning a degree": 2,
    "Associate degree (A.A., A.S., etc.)": 2,
    "Bachelor's degree (B.A., B.S., B.Eng., etc.)": 3,
    "Master's degree (M.A., M.S., M.Eng., MBA, etc.)": 4,
    "Professional degree (JD, MD, Ph.D, Ed.D, etc.)": 5,
    "Other (please specify):": 2,
}
so25["ed_ord"] = so25["EdLevel"].map(ED_ORDER).fillna(2).astype(int)

def simplify_ed(ed):
    if pd.isna(ed): return "Other"
    ed = str(ed)
    if "Bachelor" in ed: return "Bachelor's"
    if "Master" in ed or "Ph.D" in ed or "Professional" in ed: return "Graduate+"
    return "No Degree"

so25["ed_level"] = so25["EdLevel"].apply(simplify_ed)

# Experience band
def exp_band(y):
    if y <= 2:   return "0-2"
    if y <= 5:   return "3-5"
    if y <= 10:  return "6-10"
    if y <= 20:  return "11-20"
    return "20+"

so25["exp_band"] = so25["WorkExp"].apply(exp_band)

# Country grouping — top 20 countries, rest → "Other"
top_countries = so25["Country"].value_counts().head(20).index.tolist()
so25["country_grp"] = so25["Country"].where(so25["Country"].isin(top_countries), "Other")

print("Experience band distribution:")
print(so25["exp_band"].value_counts().to_string())
print("\nEducation distribution:")
print(so25["ed_level"].value_counts().to_string())
print("\nTop country groups:")
print(so25["country_grp"].value_counts().head(10).to_string())


Experience band distribution:
exp_band
11-20    4033
6-10     3130
20+      2638
3-5      1867
0-2       755

Education distribution:
ed_level
Bachelor's    5535
Graduate+     4531
No Degree     2351
Other            6

Top country groups:
country_grp
United States of America                                3828
Germany                                                 1602
United Kingdom of Great Britain and Northern Ireland    1149
France                                                   730
Canada                                                   652
Other                                                    598
India                                                    567
Netherlands                                              450
Italy                                                    408
Spain                                                    403


In [4]:
# Top 30 languages as binary features
from collections import Counter

all_langs = []
for s in so25["LanguageHaveWorkedWith"].dropna():
    all_langs.extend(s.split(";"))
top_langs = [l.strip() for l, _ in Counter(all_langs).most_common(30)]
print("Top 30 languages:", top_langs)

def col_name(lang):
    return ("lang_" + lang.lower()
            .replace("/", "_").replace(" ", "_").replace("(", "").replace(")", "")
            .replace("#", "sharp").replace("+", "plus").replace(".", "dot")
            .replace("-", "_").replace(",", "").replace("__", "_"))

lang_cols = []
for lang in top_langs:
    c = col_name(lang)
    so25[c] = so25["LanguageHaveWorkedWith"].str.contains(lang, na=False, regex=False).astype(np.int8)
    lang_cols.append(c)

print(f"\nCreated {len(lang_cols)} language feature columns")


Top 30 languages: ['JavaScript', 'HTML/CSS', 'SQL', 'Python', 'Bash/Shell (all shells)', 'TypeScript', 'C#', 'Java', 'PowerShell', 'C++', 'Go', 'C', 'PHP', 'Rust', 'Kotlin', 'Lua', 'Ruby', 'Groovy', 'Swift', 'Dart', 'Assembly', 'Visual Basic (.Net)', 'R', 'Perl', 'VBA', 'Elixir', 'GDScript', 'Scala', 'MATLAB', 'Lisp']

Created 30 language feature columns


In [5]:
# Label encode categoricals for XGBoost
le_role    = LabelEncoder().fit(so25["DevType_primary"])
le_country = LabelEncoder().fit(so25["country_grp"])

so25["role_enc"]    = le_role.transform(so25["DevType_primary"])
so25["country_enc"] = le_country.transform(so25["country_grp"])

feature_cols = ["WorkExp", "ed_ord", "role_enc", "country_enc"] + lang_cols
X = so25[feature_cols].values
y = so25["CompUSD"].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}  Test: {X_test.shape}")


Train: (9938, 34)  Test: (2485, 34)


## Train XGBoost

In [6]:
model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbosity=0,
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae  = mean_absolute_error(y_test, y_pred)
ss_res = np.sum((y_test - y_pred) ** 2)
ss_tot = np.sum((y_test - y_test.mean()) ** 2)
r2 = 1 - ss_res / ss_tot

print(f"RMSE: ${rmse:,.0f}")
print(f"MAE:  ${mae:,.0f}")
print(f"R²:   {r2:.4f}")


RMSE: $42,597
MAE:  $28,231
R²:   0.5853


In [7]:
# Feature importance plot
importances = pd.Series(model.feature_importances_, index=feature_cols)
top_imp = importances.nlargest(20)

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(top_imp.index[::-1], top_imp.values[::-1], color="#4C72B0")
ax.set_xlabel("Feature Importance (XGBoost gain)")
ax.set_title("Top 20 Features — SO 2025 Salary Model")
plt.tight_layout()
plt.savefig("figures/fig_so_model_importance.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved feature importance figure")


Saved feature importance figure


In [8]:
# Actual vs predicted scatter
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(y_test / 1_000, y_pred / 1_000, alpha=0.15, s=10, color="#4C72B0")
lim = max(y_test.max(), y_pred.max()) / 1_000
ax.plot([0, lim], [0, lim], "r--", linewidth=1.5)
ax.set_xlabel("Actual Salary ($K)")
ax.set_ylabel("Predicted Salary ($K)")
ax.set_title(f"SO 2025 Salary Model — R²={r2:.3f}, RMSE=${rmse/1000:.1f}K")
plt.tight_layout()
plt.savefig("figures/fig_so_model_pred.png", dpi=120, bbox_inches="tight")
plt.show()


## Salary Ranges Lookup Table

In [9]:
# Compute median + p25 + p75 per profile bucket
# Bucket: DevType_primary × country_grp × exp_band × ed_level
agg = (
    so25.groupby(["DevType_primary", "country_grp", "exp_band", "ed_level"])["CompUSD"]
    .agg(
        median_salary=lambda x: round(x.median(), 0),
        p25_salary=lambda x: round(x.quantile(0.25), 0),
        p75_salary=lambda x: round(x.quantile(0.75), 0),
        count="count",
    )
    .reset_index()
)
agg = agg.rename(columns={"DevType_primary": "role", "country_grp": "country"})
print(f"Salary ranges table: {len(agg):,} buckets")
print(f"Buckets with count < 30: {(agg['count'] < 30).sum():,}")
print(f"Buckets with count >= 30: {(agg['count'] >= 30).sum():,}")
print("\nSample rows:")
print(agg.sort_values("count", ascending=False).head(8).to_string())


Salary ranges table: 2,197 buckets
Buckets with count < 30: 2,135
Buckets with count >= 30: 62

Sample rows:
                       role                   country exp_band    ed_level  median_salary  p25_salary  p75_salary  count
1729  Developer, full-stack  United States of America    11-20  Bachelor's       150000.0    130000.0    190000.0    284
1738  Developer, full-stack  United States of America     6-10  Bachelor's       136000.0    109750.0    181500.0    247
1732  Developer, full-stack  United States of America      20+  Bachelor's       150000.0    125750.0    180000.0    182
1735  Developer, full-stack  United States of America      3-5  Bachelor's        99500.0     80000.0    120000.0    127
929     Developer, back-end  United States of America    11-20  Bachelor's       180000.0    149500.0    220500.0    119
1734  Developer, full-stack  United States of America      20+   No Degree       150000.0    124250.0    183000.0     98
938     Developer, back-end  United States o

In [10]:
# Also compute role-level summary (for fallback)
role_summary = (
    so25.groupby("DevType_primary")["CompUSD"]
    .agg(
        median_salary=lambda x: round(x.median(), 0),
        p25_salary=lambda x: round(x.quantile(0.25), 0),
        p75_salary=lambda x: round(x.quantile(0.75), 0),
        count="count",
    )
    .reset_index()
    .rename(columns={"DevType_primary": "role"})
)
role_summary["country"] = "All"
role_summary["exp_band"] = "All"
role_summary["ed_level"] = "All"

# Save both
agg.to_parquet(DATA_PROC / "salary_ranges.parquet", index=False)
role_summary.to_parquet(DATA_PROC / "salary_ranges_by_role.parquet", index=False)
joblib.dump(model, MODELS / "xgboost_so_salary.pkl")

print("Saved salary_ranges.parquet")
print("Saved salary_ranges_by_role.parquet")
print("Saved xgboost_so_salary.pkl")


Saved salary_ranges.parquet
Saved salary_ranges_by_role.parquet
Saved xgboost_so_salary.pkl
